In [1]:
# Kill all processess on GPU
!fuser -v /dev/nvidia* -k

In [2]:
!nvidia-smi

Sun Feb  1 21:56:37 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
from pathlib import Path
from huggingface_hub import snapshot_download

data_id = 'alxxtexxr/VRBI-Dataset'
allow_dir = 'model_unsloth_Qwen2.5-VL-3B-Instruct.data_jxie_coco_captions_train.v2'

data_dir = snapshot_download(
    repo_id=data_id,
    repo_type='dataset',
    allow_patterns=f'{allow_dir}/**',
    local_dir='data',
    local_dir_use_symlinks=False,
)
vision_data_dir = Path(data_dir) / allow_dir

print("Vision data downloaded to:", vision_data_dir)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:979: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_dir`.
For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/download#download-files-to-local-folder.
  warnings.warn(


Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

model_unsloth_Qwen2.5-VL-3B-Instruct.dat(…):   0%|          | 0.00/67.1M [00:00<?, ?B/s]

model_unsloth_Qwen2.5-VL-3B-Instruct.dat(…):   0%|          | 0.00/67.1M [00:00<?, ?B/s]

Vision data downloaded to: /content/data/model_unsloth_Qwen2.5-VL-3B-Instruct.data_jxie_coco_captions_train.v2


In [4]:
%%capture
%pip install faiss-gpu-cu12

In [6]:
from pathlib import Path
import numpy as np
import faiss

# Configuration parameters
DATA_DIR = Path('data')
DIM = 2048
K = 4096
NITER = 25
SEED = 42

# Set random seeds for reproducibility
np.random.seed(SEED)
# faiss.rand.seed(SEED) # AttributeError: 'function' object has no attribute 'seed'

# Load all batch file paths
batch_paths = [f for f in vision_data_dir.glob('*.npz')]

# Initialize KMeans (with GPU support)
kmeans = faiss.Kmeans(
    d=DIM,
    k=K,
    niter=NITER,
    verbose=True,
    gpu=True,
    seed=SEED,
    min_points_per_centroid=10,
)

In [7]:
# Collect training samples
init_buf = []

print("Collecting training samples...")

for batch_path in batch_paths:
    batch = np.load(batch_path, mmap_mode='r')
    visual_embs = batch['visual_embs'].astype(np.float32) # (32400, 2048)

    faiss.normalize_L2(visual_embs)
    init_buf.append(visual_embs)

train_data = np.vstack(init_buf)

print("Training data shape:", train_data.shape)
assert train_data.shape[0] >= K

Training data shape: (324000, 2048)


In [8]:
# Train (ONCE)
print("Training k-means...")
kmeans.train(train_data)

Training k-means...


np.float64(129627.4140625)

In [11]:
# Save codebook
codebook = kmeans.centroids
print("Codebook shape:", codebook.shape)

npy_path = f'codebook_faiss_{K}.npy'
np.save(npy_path, codebook)
print("Codebook NumPy array saved to:", npy_path)

Codebook shape: (4096, 2048)
Codebook NumPy array saved to: codebook_faiss_4096.npy


In [12]:
# Build search index (Optional)
# For later quantization
index = faiss.IndexFlatIP(DIM)  # Inner product = cosine
index.add(codebook)

index_path = f'codebook_faiss_{K}.index'
faiss.write_index(index, index_path)
print("Codebook FAISS index saved to:", index_path)

Codebook FAISS index saved to: codebook_faiss_4096.index


In [13]:
print("Min centroid norm:", np.linalg.norm(codebook, axis=1).min())

Min centroid norm: 0.52146596


In [15]:
from huggingface_hub import upload_file

upload_file(
    path_or_fileobj=npy_path,
    path_in_repo=f'{allow_dir}/{npy_path}',
    repo_id='alxxtexxr/VRBI-Dataset',
    repo_type='dataset',
)
upload_file(
    path_or_fileobj=index_path,
    path_in_repo=f'{allow_dir}/{index_path}',
    repo_id='alxxtexxr/VRBI-Dataset',
    repo_type='dataset',
)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  codebook_faiss_4096.npy     :   2%|1         |  552kB / 33.6MB            

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  codebook_faiss_4096.index   : 100%|#########9| 33.5MB / 33.6MB            

CommitInfo(commit_url='https://huggingface.co/datasets/alxxtexxr/VRBI-Dataset/commit/e7f4be0bfc4c3606c063bc4912203cb7f61006c5', commit_message='Upload model_unsloth_Qwen2.5-VL-3B-Instruct.data_jxie_coco_captions_train.v2/codebook_faiss_4096.index with huggingface_hub', commit_description='', oid='e7f4be0bfc4c3606c063bc4912203cb7f61006c5', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/alxxtexxr/VRBI-Dataset', endpoint='https://huggingface.co', repo_type='dataset', repo_id='alxxtexxr/VRBI-Dataset'), pr_revision=None, pr_num=None)